In [1]:
# Librerías

from pathlib import Path
import pandas as pd
import shutil
import re

In [2]:
BASE = Path.cwd().parent  # ajusta si el notebook no está en experiments/
PROYECTO = BASE.parent

RAW = PROYECTO / "datos" / "raw"
CURVAS = RAW / "curvas"
PERIODOS = RAW / "periodos"

print("CURVAS:", CURVAS)
print("EXISTE CURVAS:", CURVAS.exists())
print("PERIODOS:", PERIODOS)
print("EXISTE PERIODOS:", PERIODOS.exists())

CURVAS: c:\Users\adanm\Documentos\Magister en Ciencia de datos (UDLA)\TESIS\datos\raw\curvas
EXISTE CURVAS: True
PERIODOS: c:\Users\adanm\Documentos\Magister en Ciencia de datos (UDLA)\TESIS\datos\raw\periodos
EXISTE PERIODOS: True


In [3]:
all_files = list(CURVAS.rglob("*"))

print("Total elementos encontrados:", len(all_files))
print("Ejemplos:")
for f in all_files[:10]:
    print(f)

Total elementos encontrados: 520835
Ejemplos:
c:\Users\adanm\Documentos\Magister en Ciencia de datos (UDLA)\TESIS\datos\raw\curvas\OGLEIII
c:\Users\adanm\Documentos\Magister en Ciencia de datos (UDLA)\TESIS\datos\raw\curvas\OGLEIV
c:\Users\adanm\Documentos\Magister en Ciencia de datos (UDLA)\TESIS\datos\raw\curvas\OGLEIII\Anomalous cefeidas
c:\Users\adanm\Documentos\Magister en Ciencia de datos (UDLA)\TESIS\datos\raw\curvas\OGLEIII\Clasical Cefeidas
c:\Users\adanm\Documentos\Magister en Ciencia de datos (UDLA)\TESIS\datos\raw\curvas\OGLEIII\RR_Lyrae
c:\Users\adanm\Documentos\Magister en Ciencia de datos (UDLA)\TESIS\datos\raw\curvas\OGLEIII\Type II cefeidas
c:\Users\adanm\Documentos\Magister en Ciencia de datos (UDLA)\TESIS\datos\raw\curvas\OGLEIV\Anomalous cefeidas
c:\Users\adanm\Documentos\Magister en Ciencia de datos (UDLA)\TESIS\datos\raw\curvas\OGLEIV\Classical cefeidas
c:\Users\adanm\Documentos\Magister en Ciencia de datos (UDLA)\TESIS\datos\raw\curvas\OGLEIV\RR_Lyrae
c:\Users\ad

In [4]:
curvas = [f for f in all_files if f.is_file() and f.suffix.lower() == ".dat"]

print("Curvas .dat:", len(curvas))

print("\nEjemplos:")
for f in curvas[:5]:
    print(f.name)

Curvas .dat: 328998

Ejemplos:
OGLE-LMC-ACEP-001.I.dat
OGLE-LMC-ACEP-001.V.dat
OGLE-LMC-ACEP-002.I.dat
OGLE-LMC-ACEP-002.V.dat
OGLE-LMC-ACEP-003.I.dat


In [5]:
curvas_I = [f for f in curvas if ".I" in f.name]

print("Curvas banda I:", len(curvas_I))

for f in curvas_I[:5]:
    print(f.name)

Curvas banda I: 187645
OGLE-LMC-ACEP-001.I.dat
OGLE-LMC-ACEP-002.I.dat
OGLE-LMC-ACEP-003.I.dat
OGLE-LMC-ACEP-004.I.dat
OGLE-LMC-ACEP-005.I.dat


In [6]:
ej = curvas_I[0]

print("Archivo ejemplo:", ej)

Archivo ejemplo: c:\Users\adanm\Documentos\Magister en Ciencia de datos (UDLA)\TESIS\datos\raw\curvas\OGLEIII\Anomalous cefeidas\LMC\query_OGLE-LMC-ACEP-001\OGLE-LMC-ACEP-001.I.dat


In [7]:
with open(ej, encoding="utf8", errors="ignore") as f:
    for i, line in enumerate(f):
        print(line.strip())
        if i == 5:
            break

2171.88226 18.316 0.035
2183.84165 18.256 0.045
2184.84118 17.774 0.026
2191.81204 17.933 0.019
2200.82188 18.295 0.027
2206.75803 18.264 0.040


In [8]:
def parsear_curva(path):

    path = Path(path)

    nombre = path.stem.replace(".I", "")
    partes = nombre.split("-")

    # Obtener proyecto desde la ruta
    proyecto = None

    for p in path.parts:
        if p.upper() == "OGLEIII":
            proyecto = "OGLEIII"
            break
        elif p.upper() == "OGLEIV":
            proyecto = "OGLEIV"
            break

    if proyecto is None:
        proyecto = "DESCONOCIDO"

    return {
        "proyecto": proyecto,
        "region": partes[1],
        "tipo": partes[2],
        "id": partes[3],
        "base": nombre
    }

In [9]:
parsear_curva(curvas_I[55])

{'proyecto': 'OGLEIII',
 'region': 'LMC',
 'tipo': 'ACEP',
 'id': '056',
 'base': 'OGLE-LMC-ACEP-056'}

In [10]:
def contar_obs(path):
    with open(path, encoding="utf8", errors="ignore") as f:
        return sum(1 for l in f if l.strip())

In [11]:
contar_obs(curvas_I[0])

362

In [38]:
rows = []

for f in curvas_I:
    meta = parsear_curva(f.name)
    n_obs = contar_obs(f)

    rows.append({
        **meta,
        "ruta": str(f),
        "n_obs": n_obs
    })

catalogo_curvas = pd.DataFrame(rows)

catalogo_curvas.head()

KeyboardInterrupt: 

In [13]:
print("Total curvas:", len(catalogo_curvas))
print("\nDistribución por tipo:")
print(catalogo_curvas["tipo"].value_counts())

print("\nDistribución por región:")
print(catalogo_curvas["region"].value_counts())

Total curvas: 187645

Distribución por tipo:
tipo
RRLYR    167037
CEP       17757
T2CEP      2503
ACEP        348
Name: count, dtype: int64

Distribución por región:
region
BLG    83070
LMC    73882
SMC    17106
GD     13587
Name: count, dtype: int64


In [39]:
MIN_OBS = 300

curvas_limpias = catalogo_curvas[
    catalogo_curvas["n_obs"] >= MIN_OBS
].copy()

print("Curvas con >=300 obs:", len(curvas_limpias))
curvas_limpias.head()

Curvas con >=300 obs: 118135


,proyecto,region,tipo,id,base,ruta,n_obs
0,OGLEIII,LMC,ACEP,001,OGLE-LMC-ACEP-001,c:\Users\adanm\Documentos\Magister en Ciencia ...,362
1,OGLEIII,LMC,ACEP,002,OGLE-LMC-ACEP-002,c:\Users\adanm\Documentos\Magister en Ciencia ...,380
2,OGLEIII,LMC,ACEP,003,OGLE-LMC-ACEP-003,c:\Users\adanm\Documentos\Magister en Ciencia ...,725
3,OGLEIII,LMC,ACEP,004,OGLE-LMC-ACEP-004,c:\Users\adanm\Documentos\Magister en Ciencia ...,361
4,OGLEIII,LMC,ACEP,005,OGLE-LMC-ACEP-005,c:\Users\adanm\Documentos\Magister en Ciencia ...,370


In [25]:
f = curvas_I[0]

print(f)
print()
print(f.parts)

c:\Users\adanm\Documentos\Magister en Ciencia de datos (UDLA)\TESIS\datos\raw\curvas\OGLEIII\Anomalous cefeidas\LMC\query_OGLE-LMC-ACEP-001\OGLE-LMC-ACEP-001.I.dat

('c:\\', 'Users', 'adanm', 'Documentos', 'Magister en Ciencia de datos (UDLA)', 'TESIS', 'datos', 'raw', 'curvas', 'OGLEIII', 'Anomalous cefeidas', 'LMC', 'query_OGLE-LMC-ACEP-001', 'OGLE-LMC-ACEP-001.I.dat')


In [26]:
print(type(curvas_I[0]))
print(curvas_I[0])

print()

print(type(curvas_I[0].name))
print(curvas_I[0].name)

<class 'pathlib.WindowsPath'>
c:\Users\adanm\Documentos\Magister en Ciencia de datos (UDLA)\TESIS\datos\raw\curvas\OGLEIII\Anomalous cefeidas\LMC\query_OGLE-LMC-ACEP-001\OGLE-LMC-ACEP-001.I.dat

<class 'str'>
OGLE-LMC-ACEP-001.I.dat


In [15]:
periodos_files = [f for f in PERIODOS.rglob("*") if f.is_file()]

print("Archivos de períodos:", len(periodos_files))

periodos_files[:5]

Archivos de períodos: 92


[WindowsPath('c:/Users/adanm/Documentos/Magister en Ciencia de datos (UDLA)/TESIS/datos/raw/periodos/OGLEIII/BLG/cep1O.dat'),
 WindowsPath('c:/Users/adanm/Documentos/Magister en Ciencia de datos (UDLA)/TESIS/datos/raw/periodos/OGLEIII/BLG/cep1O2O.dat'),
 WindowsPath('c:/Users/adanm/Documentos/Magister en Ciencia de datos (UDLA)/TESIS/datos/raw/periodos/OGLEIII/BLG/cep1O2O3O.dat'),
 WindowsPath('c:/Users/adanm/Documentos/Magister en Ciencia de datos (UDLA)/TESIS/datos/raw/periodos/OGLEIII/BLG/cepF.dat'),
 WindowsPath('c:/Users/adanm/Documentos/Magister en Ciencia de datos (UDLA)/TESIS/datos/raw/periodos/OGLEIII/BLG/cepF1O.dat')]

In [27]:
f = curvas_I[0]

meta = parsear_curva(f)

print(meta)

{'proyecto': 'OGLEIII', 'region': 'LMC', 'tipo': 'ACEP', 'id': '001', 'base': 'OGLE-LMC-ACEP-001'}


In [28]:
rows = []

for f in curvas_I:
    meta = parsear_curva(f)
    n_obs = contar_obs(f)

    rows.append({
        **meta,
        "ruta": str(f),
        "n_obs": n_obs
    })

catalogo_curvas = pd.DataFrame(rows)

In [29]:
ej = periodos_files[0]

print("Archivo:", ej)

with open(ej, encoding="utf8", errors="ignore") as f:
    for i, line in enumerate(f):
        print(line.strip())
        if i == 5:
            break

Archivo: c:\Users\adanm\Documentos\Magister en Ciencia de datos (UDLA)\TESIS\datos\raw\periodos\OGLEIII\BLG\cep1O.dat
OGLE-BLG-CEP-15  15.934 17.772   0.5299659 0.0000004  5000.22287  0.178  0.132 3.663  0.040 0.938
OGLE-BLG-CEP-20  14.689 16.073   0.4406806 0.0000002  5000.37252  0.130  0.070 2.519  0.020 5.750
OGLE-BLG-CEP-24  15.639 16.816   0.3554320 0.0000002  5000.18232  0.359  0.102 3.680  0.043 0.482
OGLE-BLG-CEP-27  14.644 15.673   0.2965307 0.0000002  5000.07980  0.106  0.124 3.954  0.017 1.653


In [30]:
rows = []

for archivo in periodos_files:

    # Determinar de qué proyecto proviene
    if "OGLEIII" in str(archivo):
        proyecto = "OGLEIII"
    elif "OGLEIV" in str(archivo):
        proyecto = "OGLEIV"
    else:
        proyecto = "DESCONOCIDO"

    with open(archivo, encoding="utf8", errors="ignore") as f:

        for linea in f:

            linea = linea.strip()

            if not linea:
                continue

            partes = linea.split()

            # Comprobación básica
            if len(partes) < 4:
                continue

            nombre = partes[0]
            periodo = float(partes[3])

            rows.append({
                "id": nombre,
                "proyecto": proyecto,
                "periodo": periodo
            })

catalogo_periodos = pd.DataFrame(rows)

print("Períodos cargados:", len(catalogo_periodos))
catalogo_periodos.head()

Períodos cargados: 195541


,id,proyecto,periodo
0,OGLE-BLG-CEP-15,OGLEIII,0.529966
1,OGLE-BLG-CEP-20,OGLEIII,0.440681
2,OGLE-BLG-CEP-24,OGLEIII,0.355432
3,OGLE-BLG-CEP-27,OGLEIII,0.296531
4,OGLE-BLG-CEP-04,OGLEIII,0.240046


In [47]:
duplicados = catalogo_periodos.duplicated(
    subset=["proyecto", "id"],
    keep=False
)

catalogo_periodos[duplicados].sort_values(["id", "proyecto"]).head(20)

,id,proyecto,periodo
123062,OGLE-GAL-ACEP-091,OGLEIV,0.674532
123133,OGLE-GAL-ACEP-091,OGLEIV,0.928691


In [46]:
catalogo_periodos.loc[
    catalogo_periodos["id"] == "OGLE-GAL-ACEP-091"
]

,id,proyecto,periodo
123062,OGLE-GAL-ACEP-091,OGLEIV,0.674532
123133,OGLE-GAL-ACEP-091,OGLEIV,0.928691


In [44]:
rows.append({
    "id": nombre,
    "proyecto": proyecto,
    "periodo": periodo,
    "archivo_periodos": archivo.name
})

In [45]:
catalogo_periodos.loc[
    catalogo_periodos["id"] == "OGLE-GAL-ACEP-091"
]

,id,proyecto,periodo
123062,OGLE-GAL-ACEP-091,OGLEIV,0.674532
123133,OGLE-GAL-ACEP-091,OGLEIV,0.928691


In [42]:
duplicados = catalogo_periodos.duplicated(
    subset=["proyecto", "id"],
    keep=False
)

catalogo_periodos[duplicados]["id"].nunique()

1

In [41]:
catalogo_periodos[duplicados]

,id,proyecto,periodo
123062,OGLE-GAL-ACEP-091,OGLEIV,0.674532
123133,OGLE-GAL-ACEP-091,OGLEIV,0.928691


In [40]:
curvas_limpias[["proyecto"]].drop_duplicates()

,proyecto
0,OGLEIII
50161,OGLEIV


In [48]:
rows = []

for archivo in periodos_files:

    # Proyecto desde la ruta
    proyecto = Path(archivo).parts[
        Path(archivo).parts.index("periodos") + 1
    ]

    with open(archivo, encoding="utf8", errors="ignore") as f:

        for linea in f:

            linea = linea.strip()

            if not linea:
                continue

            partes = linea.split()

            if len(partes) < 4:
                continue

            rows.append({
                "proyecto": proyecto,
                "id": partes[0],
                "periodo": float(partes[3]),
                "archivo": archivo.name
            })

catalogo_periodos = pd.DataFrame(rows)

catalogo_periodos.head()

,proyecto,id,periodo,archivo
0,OGLEIII,OGLE-BLG-CEP-15,0.529966,cep1O.dat
1,OGLEIII,OGLE-BLG-CEP-20,0.440681,cep1O.dat
2,OGLEIII,OGLE-BLG-CEP-24,0.355432,cep1O.dat
3,OGLEIII,OGLE-BLG-CEP-27,0.296531,cep1O.dat
4,OGLEIII,OGLE-BLG-CEP-04,0.240046,cep1O2O.dat


In [49]:
catalogo_periodos["proyecto"].value_counts()

proyecto
OGLEIV     142537
OGLEIII     53004
Name: count, dtype: int64

In [50]:
catalogo_curvas["clave"] = (
    catalogo_curvas["proyecto"] + "_" +
    catalogo_curvas["base"]
)

In [51]:
catalogo_periodos["clave"] = (
    catalogo_periodos["proyecto"] + "_" +
    catalogo_periodos["id"]
)

In [52]:
rows = []

for archivo in periodos_files:

    path = Path(archivo)

    # Proyecto (OGLEIII u OGLEIV)
    proyecto = path.parts[path.parts.index("periodos") + 1]

    with open(path, encoding="utf8", errors="ignore") as f:

        for linea in f:

            linea = linea.strip()

            if not linea:
                continue

            partes = linea.split()

            if len(partes) < 4:
                continue

            nombre = partes[0]
            periodo = float(partes[3])

            nombre_partes = nombre.split("-")

            if len(nombre_partes) != 4:
                continue

            rows.append({
                "proyecto": proyecto,
                "region": nombre_partes[1],
                "tipo": nombre_partes[2],
                "id": nombre_partes[3],
                "base": nombre,
                "periodo": periodo,
                "archivo_periodos": path.name,
                "ruta_periodos": str(path)
            })

catalogo_periodos = pd.DataFrame(rows)

print("Registros:", len(catalogo_periodos))
catalogo_periodos.head()

Registros: 195541


,proyecto,region,tipo,id,base,periodo,archivo_periodos,ruta_periodos
0,OGLEIII,BLG,CEP,15,OGLE-BLG-CEP-15,0.529966,cep1O.dat,c:\Users\adanm\Documentos\Magister en Ciencia ...
1,OGLEIII,BLG,CEP,20,OGLE-BLG-CEP-20,0.440681,cep1O.dat,c:\Users\adanm\Documentos\Magister en Ciencia ...
2,OGLEIII,BLG,CEP,24,OGLE-BLG-CEP-24,0.355432,cep1O.dat,c:\Users\adanm\Documentos\Magister en Ciencia ...
3,OGLEIII,BLG,CEP,27,OGLE-BLG-CEP-27,0.296531,cep1O.dat,c:\Users\adanm\Documentos\Magister en Ciencia ...
4,OGLEIII,BLG,CEP,04,OGLE-BLG-CEP-04,0.240046,cep1O2O.dat,c:\Users\adanm\Documentos\Magister en Ciencia ...


In [53]:
print("Proyectos")
display(catalogo_periodos["proyecto"].value_counts())

print("\nTipos")
display(catalogo_periodos["tipo"].value_counts())

print("\nRegiones")
display(catalogo_periodos["region"].value_counts())

Proyectos


proyecto
OGLEIV     142537
OGLEIII     53004
Name: count, dtype: int64


Tipos


tipo
RRLYR    172735
CEP       19745
T2CEP      2588
ACEP        473
Name: count, dtype: int64


Regiones


region
BLG    87255
LMC    75190
SMC    19112
GD     13864
GAL      120
Name: count, dtype: int64

In [54]:
duplicados = catalogo_periodos.duplicated(
    subset=["proyecto", "base"],
    keep=False
)

print("Duplicados:", duplicados.sum())

catalogo_periodos.loc[duplicados].sort_values(
    ["base", "proyecto"]
)

Duplicados: 2


,proyecto,region,tipo,id,base,periodo,archivo_periodos,ruta_periodos
123062,OGLEIV,GAL,ACEP,091,OGLE-GAL-ACEP-091,0.674532,acep1O.dat,c:\Users\adanm\Documentos\Magister en Ciencia ...
123133,OGLEIV,GAL,ACEP,091,OGLE-GAL-ACEP-091,0.928691,acepF.dat,c:\Users\adanm\Documentos\Magister en Ciencia ...


In [55]:
catalogo_periodos.to_csv(
    BASE / "catalogo_periodos.csv",
    index=False
)

In [56]:
catalogo_final = catalogo_curvas.merge(
    catalogo_periodos,
    on=["proyecto", "base"],
    how="left",
    suffixes=("_curva", "_periodo")
)

In [57]:
catalogo_final = catalogo_curvas.merge(
    catalogo_periodos[
        ["proyecto", "base", "periodo"]
    ],
    on=["proyecto", "base"],
    how="left"
)

In [58]:
catalogo_final.head()

,proyecto,region,tipo,id,base,ruta,n_obs,clave,periodo
0,OGLEIII,LMC,ACEP,001,OGLE-LMC-ACEP-001,c:\Users\adanm\Documentos\Magister en Ciencia ...,362,OGLEIII_OGLE-LMC-ACEP-001,0.850233
1,OGLEIII,LMC,ACEP,002,OGLE-LMC-ACEP-002,c:\Users\adanm\Documentos\Magister en Ciencia ...,380,OGLEIII_OGLE-LMC-ACEP-002,0.976640
2,OGLEIII,LMC,ACEP,003,OGLE-LMC-ACEP-003,c:\Users\adanm\Documentos\Magister en Ciencia ...,725,OGLEIII_OGLE-LMC-ACEP-003,0.381780
3,OGLEIII,LMC,ACEP,004,OGLE-LMC-ACEP-004,c:\Users\adanm\Documentos\Magister en Ciencia ...,361,OGLEIII_OGLE-LMC-ACEP-004,1.861827
4,OGLEIII,LMC,ACEP,005,OGLE-LMC-ACEP-005,c:\Users\adanm\Documentos\Magister en Ciencia ...,370,OGLEIII_OGLE-LMC-ACEP-005,0.932159


In [59]:
SALIDA = BASE / "catalogo_final.csv"

catalogo_final.to_csv(
    SALIDA,
    index=False
)

print(f"Catálogo guardado en:\n{SALIDA}")

Catálogo guardado en:
c:\Users\adanm\Documentos\Magister en Ciencia de datos (UDLA)\TESIS\experimentos\catalogo_final.csv


In [60]:
len(catalogo_curvas)

187645

In [61]:
len(curvas_limpias)

118135

In [62]:
len(catalogo_final)

187645

In [63]:
catalogo_final = curvas_limpias.merge(
    catalogo_periodos[
        ["proyecto", "base", "periodo"]
    ],
    on=["proyecto", "base"],
    how="left"
)

In [64]:
catalogo_final["n_obs"].min()

np.int64(300)

In [65]:
(catalogo_final["n_obs"] < 300).sum()

np.int64(0)

In [66]:
SALIDA = BASE / "catalogo_final.csv"

catalogo_final.to_csv(
    SALIDA,
    index=False
)

print(f"Catálogo guardado en:\n{SALIDA}")

Catálogo guardado en:
c:\Users\adanm\Documentos\Magister en Ciencia de datos (UDLA)\TESIS\experimentos\catalogo_final.csv
